In [ ]:
head -n 2200 data/iowa_sales_finetuning_2500_rich_unique.jsonl > data/train.jsonl
sed -n '2201,2350p' data/iowa_sales_finetuning_2500_rich_unique.jsonl > data/valid.jsonl
sed -n '2351,2500p' data/iowa_sales_finetuning_2500_rich_unique.jsonl > data/test.jsonl

In [ ]:
#!/bin/bash

set -e

echo "Starting training..."

date

mlx_lm lora \
  --model ./Qwen3-8B-MLX-bf16 \
  --train \
  --data ./data \
  --fine-tune-type lora \
  --adapter-path ./Qwen3-8B-MLX-bf16-iowa-lora-v1 \
  --batch-size 1 \
  --iters 2500 \
  --learning-rate 1e-4 \
  --num-layers 16 \
  --mask-prompt \
  --save-every 500 \
  --max-seq-length 1024

echo "Training finished."

date

In [2]:
from mlx_lm import load, generate
import gc

QUESTIONS = ["What is the business grain of iowa_sales?", "List all the relationships for iowa_sales table. Provide * table name * pk and fk column name"]

In [ ]:
for q in QUESTIONS:
    messages = [{"role": "user", "content": f"/no_think {q}"}]

    # Base model
    base_model, base_tokenizer = load("./Qwen3-8B-MLX-bf16")

    prompt = base_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    print("\n========== BASE MODEL ==========\n")

    base_response = generate(
        base_model,
        base_tokenizer,
        prompt=prompt,
        max_tokens=200,
        verbose=False
    )

    print(f"Question: {q}")
    print(f"Response: {base_response}")


========== BASE MODEL ==========

Question: What is the business grain of iowa_sales?
Response: <think>

</think>

The term **"iowa_sales"** is not a standard or widely recognized business term or entity in the context of business analytics, finance, or general business operations. It could refer to a few different things depending on the context:

### 1. **A Dataset or Database**
- **iowa_sales** might be the name of a dataset or database that contains sales data related to Iowa (a U.S. state). This could be used in data analysis, machine learning, or business intelligence projects.
- For example, it could be a dataset used in a Kaggle competition or a university project that includes sales figures, customer data, product information, etc., specific to Iowa.

### 2. **A Business or Company**
- It could be the name of a company or business based in Iowa that deals with sales, such as a retail chain, agricultural product sales, or a local business.
- However, without more context, it's

0

In [ ]:
for q in QUESTIONS:
    messages = [{"role": "user", "content": f"/no_think {q}"}]
    
    # Fine-tuned model
    ft_model, ft_tokenizer = load(
        "./Qwen3-8B-MLX-bf16",
        adapter_path="./Qwen3-8B-MLX-bf16-iowa-lora-v1",
    )

    prompt = ft_tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )

    print("\n========== FINE-TUNED MODEL ==========\n")

    ft_response = generate(
        ft_model, 
        ft_tokenizer, 
        prompt=prompt, 
        max_tokens=200, 
        verbose=False
    )

    print(f"Question: {q}")
    print(f"Response: {ft_response}")


========== FINE-TUNED MODEL ==========

Question: What is the business grain of iowa_sales?
Response: <think>

</think>

iowa_sales is a fact table. It is the central sales transaction table and its grain is one invoice item line.

========== FINE-TUNED MODEL ==========

Question: List all the relationships for iowa_sales table. Provide * table name * pk and fk column name
Response: <think>

</think>

For the iowa_sales table, the relationships are as follows:

1. iowa_sales.ForeignKey = iowa_category.CategoryID
   - iowa_sales.CategoryID is foreign key to iowa_category.CategoryID

2. iowa_sales.ForeignKey = iowa_vendor.VendorID
   - iowa_sales.VendorID is foreign key to iowa_vendor.VendorID

3. iowa_sales.ForeignKey = iowa_item.ItemID
   - iowa_sales.ItemID is foreign key to iowa_item.ItemID

4. iowa_sales.ForeignKey = iowa_store.StoreID
   - iowa_sales.StoreID is foreign key to iowa_store.StoreID

5. iowa_sales.ForeignKey = iowa_vendor.VendorID (if it is a relationship to iowa_vendo

0